In [ ]:
import os
import mne
import numpy as np
import pandas as pd
from scipy.signal import welch

# the paired files for one recording
psg_path  = '../physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4201E0-PSG.edf'
hyp_path  = '../physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4201EC-Hypnogram.edf'

# load signals
raw = mne.io.read_raw_edf(psg_path, preload=True)

# load the hypnogram annotations and attach them to the raw object
annotations = mne.read_annotations(hyp_path)
raw.set_annotations(annotations)

print(raw.annotations)
print('duration (s):', raw.times[-1])

Extracting EDF parameters from ../physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/SC4201E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 8411999  =      0.000 ... 84119.990 secs...


/var/folders/hs/4hdxpkyn0g5274r3582nqd_h0000gn/T/ipykernel_6573/4187551074.py:10: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
/var/folders/hs/4hdxpkyn0g5274r3582nqd_h0000gn/T/ipykernel_6573/4187551074.py:10: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
/var/folders/hs/4hdxpkyn0g5274r3582nqd_h0000gn/T/ipykernel_6573/4187551074.py:10: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
/var/folders/hs/4hdxpkyn0g5274r3582nqd_h0000gn/T/ipykernel_6573/4187551074.py:14: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annotations)


<Annotations | 56 segments: Movement time (1), Sleep stage 1 (15), Sleep ...>
duration (s): 84119.99


In [2]:
# the channels we care about for sleep staging
keep_channels = ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'EMG submental']

# drop the rest (respiration, temperature, event marker — not used for staging)
raw.pick(keep_channels)

# bandpass filter: 0.5-30 Hz removes slow drift + high-freq noise
raw.filter(l_freq=0.5, h_freq=30)

print('channels now:', raw.ch_names)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 661 samples (6.610 s)

channels now: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'EMG submental']


In [3]:
# our mapping: stage name -> integer label
# note: merge stage 3 AND 4 into the same class (N3), drop ? and Movement
stage_map = {
    'Sleep stage W': 0,
    'Sleep stage 1': 1,
    'Sleep stage 2': 2,
    'Sleep stage 3': 3,
    'Sleep stage 4': 3,   # <-- 4 merged into 3 (N3)
    'Sleep stage R': 4,
}

# convert annotations to events using this mapping
events, event_id = mne.events_from_annotations(raw, event_id=stage_map, chunk_duration=30.0)

print('event_id used:', event_id)
print('number of events:', len(events))
print('first few events:\n', events[:5])

Used Annotations descriptions: ['Sleep stage 1', 'Sleep stage 2', 'Sleep stage 3', 'Sleep stage R', 'Sleep stage W']
event_id used: {'Sleep stage 1': 1, 'Sleep stage 2': 2, 'Sleep stage 3': 3, 'Sleep stage R': 4, 'Sleep stage W': 0}
number of events: 2803
first few events:
 [[    0     0     0]
 [ 3000     0     0]
 [ 6000     0     0]
 [ 9000     0     0]
 [12000     0     0]]


In [4]:
labels_col = events[:, 2]   # the third column = stage label
unique, counts = np.unique(labels_col, return_counts=True)

stage_names = {0: 'W', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}
for u, c in zip(unique, counts):
    print(f'{stage_names[u]:4s} (label {u}): {c}')

W    (label 0): 2044
N1   (label 1): 39
N2   (label 2): 539
N3   (label 3): 4
REM  (label 4): 177


In [5]:
# find the sleep period from the annotations (non-Wake stages)
sleep_stages = ['Sleep stage 1', 'Sleep stage 2', 'Sleep stage 3',
                'Sleep stage 4', 'Sleep stage R']

# onsets & durations of all annotations
onsets = raw.annotations.onset
durations = raw.annotations.duration
descriptions = raw.annotations.description

# find first and last non-wake annotation
sleep_mask = np.isin(descriptions, sleep_stages)
sleep_onsets = onsets[sleep_mask]
sleep_ends = (onsets + durations)[sleep_mask]

first_sleep = sleep_onsets.min()
last_sleep = sleep_ends.max()

# add a 30-minute wake buffer on each side
buffer = 30 * 60   # 30 minutes in seconds
crop_start = max(0, first_sleep - buffer)
crop_end = min(raw.times[-1], last_sleep + buffer)

print(f'first sleep at: {first_sleep:.0f}s ({first_sleep/3600:.1f}h)')
print(f'last sleep at:  {last_sleep:.0f}s ({last_sleep/3600:.1f}h)')
print(f'cropping to: {crop_start:.0f}s -> {crop_end:.0f}s')
print(f'kept duration: {(crop_end-crop_start)/3600:.1f}h (was {raw.times[-1]/3600:.1f}h)')

first sleep at: 25350s (7.0h)
last sleep at:  52440s (14.6h)
cropping to: 23550s -> 54240s
kept duration: 8.5h (was 23.4h)


In [6]:
# crop the recording to the sleep period + buffers
raw.crop(tmin=crop_start, tmax=crop_end)

# re-run the epoching on the cropped data
events, event_id = mne.events_from_annotations(raw, event_id=stage_map, chunk_duration=30.0)

# recount
labels_col = events[:, 2]
unique, counts = np.unique(labels_col, return_counts=True)
stage_names = {0: 'W', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}
print('after trimming:')
for u, c in zip(unique, counts):
    print(f'{stage_names[u]:4s} (label {u}): {c}')
print('total epochs:', len(events))

Used Annotations descriptions: ['Sleep stage 1', 'Sleep stage 2', 'Sleep stage 3', 'Sleep stage R', 'Sleep stage W']
after trimming:
W    (label 0): 263
N1   (label 1): 39
N2   (label 2): 539
N3   (label 3): 4
REM  (label 4): 177
total epochs: 1022


In [7]:
# build epochs: extract the actual signal for each 30s window
epochs = mne.Epochs(
    raw, events, event_id=event_id,
    tmin=0., tmax=30. - 1/100,   # 30 seconds (minus one sample so it's exactly 3000)
    baseline=None, preload=True
)

print(epochs)
print('epochs data shape:', epochs.get_data().shape)

Not setting metadata
1022 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1022 events and 3000 original time points ...
0 bad epochs dropped
<Epochs | 1022 events (all good), 0 – 29.99 s (baseline off), ~93.6 MiB, data loaded,
 'Sleep stage 1': 39
 'Sleep stage 2': 539
 'Sleep stage 3': 4
 'Sleep stage R': 177
 'Sleep stage W': 263>
epochs data shape: (1022, 4, 3000)


In [ ]:
# get the full data array and labels
data = epochs.get_data()          # shape (1022, 4, 3000)
labels = epochs.events[:, 2]      # the stage label per epoch

print('data:', data.shape, ' labels:', labels.shape)

BANDS = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30)}
fs = 100

def epoch_band_powers(epoch, fs=100):
    """epoch: (n_channels, 3000). Returns a flat dict of band powers per channel."""
    ch_names = ['fpz', 'pz', 'eog', 'emg']   # short names for the 4 channels
    feats = {}
    for ci, ch in enumerate(ch_names):
        sig = epoch[ci]                       # this channel's 3000 samples
        freqs, psd = welch(sig, fs=fs, nperseg=fs*2)
        for band, (low, high) in BANDS.items():
            mask = (freqs >= low) & (freqs < high)
            feats[f'{ch}_{band}'] = np.sum(psd[mask])
    return feats

# test on ONE epoch
f0 = epoch_band_powers(data[0])
print('number of features:', len(f0))
for k, v in f0.items():
    print(f'  {k}: {v:.2e}')

data: (1022, 4, 3000)  labels: (1022,)
number of features: 16
  fpz_delta: 1.23e-09
  fpz_theta: 1.07e-10
  fpz_alpha: 2.97e-11
  fpz_beta: 1.03e-10
  pz_delta: 1.83e-10
  pz_theta: 1.61e-11
  pz_alpha: 1.79e-11
  pz_beta: 5.85e-11
  eog_delta: 1.62e-08
  eog_theta: 6.74e-10
  eog_alpha: 1.32e-10
  eog_beta: 8.30e-11
  emg_delta: 1.80e-15
  emg_theta: 1.01e-21
  emg_alpha: 2.27e-23
  emg_beta: 1.96e-24


In [9]:
# extract features for every epoch
feature_rows = []
for i in range(len(data)):
    feature_rows.append(epoch_band_powers(data[i]))

X = pd.DataFrame(feature_rows)
y = pd.Series(labels, name='stage')

print('X shape:', X.shape)
print('y shape:', y.shape)
X.head()

X shape: (1022, 16)
y shape: (1022,)


,fpz_delta,fpz_theta,fpz_alpha,fpz_beta,pz_delta,pz_theta,pz_alpha,pz_beta,eog_delta,eog_theta,eog_alpha,eog_beta,emg_delta,emg_theta,emg_alpha,emg_beta
0,1.227295e-09,1.072034e-10,2.973403e-11,1.025688e-10,1.830835e-10,1.608193e-11,1.792495e-11,5.850236e-11,1.618860e-08,6.740651e-10,1.317697e-10,8.296282e-11,1.795929e-15,1.008819e-21,2.271363e-23,1.962392e-24
1,1.295424e-09,7.665522e-11,2.184330e-11,3.625837e-11,1.829337e-10,1.386411e-11,1.730749e-11,5.530692e-11,1.192586e-08,4.133599e-10,9.121777e-11,1.200089e-10,2.338310e-15,2.077346e-21,4.753366e-23,4.120724e-24
2,8.885690e-10,1.076562e-10,1.204028e-11,2.947664e-11,6.805549e-11,6.803097e-12,8.287258e-12,2.909886e-11,1.119298e-08,6.207531e-10,1.343739e-10,1.870133e-10,1.496390e-15,1.593089e-21,3.644610e-23,3.159409e-24
3,1.054658e-09,9.749087e-11,1.805861e-11,2.863789e-11,5.337786e-11,8.561880e-12,8.703739e-12,3.330077e-11,1.137577e-08,6.455543e-10,1.413014e-10,8.685845e-11,1.702086e-15,2.289693e-21,5.212124e-23,4.513242e-24
4,1.830717e-09,1.672981e-10,2.985289e-11,4.832254e-11,1.402452e-10,1.277573e-11,1.268860e-11,5.239874e-11,1.505394e-08,5.050732e-10,9.114309e-11,1.293096e-10,8.711715e-16,9.683606e-22,2.212222e-23,1.917122e-24


In [ ]:
def parse_subject(psg_filename):
    """SC4201E0-PSG.edf -> subject=20, night=1"""
    base = os.path.basename(psg_filename)   # SC4201E0-PSG.edf
    # SC4 [ss] [N] E0 ...  -> chars 3-4 = subject, char 5 = night
    subject = int(base[3:5])
    night = int(base[5])
    return subject, night

# test it
subj, night = parse_subject('SC4201E0-PSG.edf')
print(f'subject: {subj}, night: {night}')

subject: 20, night: 1
